# IC Cache for Evaluation Tool

The Reference Implementation (RI) EvaluationTool replaces the basic `evaluate()` function in `maite.workflows` with an object which can associate multiple evaluations with a single backend cache.

The RI team has implemented predict and evaluate caches using JSON serialization.  However, the specific implementation differs between Image Classification (IC) and Object Detection (OD) tasks due to differences in the data structure of `Targets`.

This notebook is a simple proof-of-concept for the IC caching use case.


## Assemble Simple Fake Dataset, Fake Model, and Accuracy Metric
First, we create a fake IC dataset which contains 6 images of two classes.

In [1]:
from maite._internals.protocols import ArrayLike
import maite.protocols.image_classification as ic
from torch import Tensor

class FakeDataset(ic.Dataset):
    def __init__(self,
               images = [ Tensor(1,1,1) for _ in range(6) ], # 6 fake images
               targets = [ Tensor([(n-1)%2, n%2]) for n in range(6) ], # alternate tagging as class 0 and 1
               metadata = [ {"id": i} for i in range(6) ] # each image gets an ID
    ):
        self.images = images
        self.targets = targets
        self.metadata = metadata
    def __getitem__(self, ind:int):
      return (self.images[ind],self.targets[ind],self.metadata[ind])

    def __len__(self) -> int:
      return len(self.images)

dataset = FakeDataset()

Then we set up a fake model.  Here are its main attributes for demo purposes:
* It guesses that every image is class 0.  So out of the balanced cat/dog dataset above, it will be 50% correct.
* It pauses for 1 second per prediction batch, so we can demonstrate the improved execution time when loading from the cache.

In [2]:
import time

class FakeModel(ic.Model):
    def __call__(self, input) -> list[Tensor]:
        time.sleep(1)
        return [ Tensor([0.9,0.1]) for _ in range(len(input)) ]

model = FakeModel()

Lastly, we bring in a (real, simple) accuracy metric.

In [3]:
from jatic_ri.image_classification.metrics import accuracy_multiclass_torch_metric_factory

accuracy = accuracy_multiclass_torch_metric_factory(num_classes=2)

## Run Simple Evaluation

We create a evaluation tool object backed by an IC cache.

> Note that the flag on the SimpleRICacheIC object `compress_json=False` dictates that cache files will be written in an uncompressed (and, therefore, human readable) format on disk.  This is helpful for demo and debugging.  The default behavior is to compress the JSON for storage efficiency.

In [4]:
import os
from jatic_ri.util.cache import SimpleRICacheIC
from jatic_ri.util.evaluation import EvaluationTool

current_dir = os.getcwd()
tmp_dir = os.path.join(current_dir, 'cache_dir')
evaluationtool = EvaluationTool(SimpleRICacheIC(cache_root_dir=tmp_dir, compress_json=False))

Next we use a simple loop to run the exact same evaluation code twice.

Both times, the accuracy is 50% (as expected given our fake setup).

But the first run takes approximately 6 seconds while the second "run" responds instantly because the previous result was in cache.
> NOTE: This creates a cache directory in your filesystem at ./cache_dir

In [ ]:
for i in [1,2]:
  start_time = time.time()
  results, _, _ = evaluationtool.evaluate(model=model,dataset=dataset,metric=accuracy,model_id="fake_model",dataset_id="fake_dataset",metric_id="accuracy")
  print(f"Run {i} complete.  Accuracy is {results['accuracy']}. Elapsed time was {time.time() - start_time:.2f} seconds")

## But What If we Want Another Metric?

If you inspect your `./cache_dir` now, you will see two files.

`fake_model_fake_dataset_1.json` contains only the *computed predictions*.
`fake_model_fake_dataset_accuracy_1.json` contains the predictions *and the calculated metric(s)*.

> Note that the `1` at the end of the cache file names corresponds to the batch size of 1.  If a prediction sequence is run with the same model and dataset but different batch size, we will cache that separately in case the different batch sizes produce meaningfully different results.

This allows us to run different metrics with the same (cached) prediction set for a Model-Dataset pair.

For example, we can create a new (real but simple) F1 Score metric and calculate it for the same dataset and model.
* The result will compute very quickly because the cached Model-Dataset file lets us bypass the prediction computing (and the 1 second delays built into our fake mdoel)
* A third file will now be present in our `./cache_dir` since F1 Score calculation is now cached as well.

In [ ]:
from jatic_ri.image_classification.metrics import f1score_multiclass_torch_metric_factory

f1score = f1score_multiclass_torch_metric_factory(num_classes=2)

start_time = time.time()
results, _, _ = evaluationtool.evaluate(model=model,dataset=dataset,metric=f1score,model_id="fake_model",dataset_id="fake_dataset",metric_id="f1_score")
print(f"Run 3 complete.  F1Score is {results}. Elapsed time was {time.time() - start_time:.2f} seconds")